
# LLM Evaluation Pipeline (Math Reasoning)

This notebook provides a **ready-to-run pipeline** to run an LLM on a math reasoning dataset (AIMathsOlympiadData) and store the answer in a separate column in the df.

## What this notebook does:
- Loads dataset
- Formats prompts
- Runs LLM (placeholder / OpenAI-ready)
---

## ⚠️ Setup Instructions
1. Download dataset from Kaggle:
   https://www.kaggle.com/datasets/satyaprakashshukl/aimathsolympiaddata

2. Place the CSV/parquet, etc in:
   `data/file.csv`


In [10]:
import pandas as pd
import numpy as np
import random
import os
from typing import Dict
from dotenv import load_dotenv

# .envrc format is 'export KEY="VALUE"', so we use a custom function to load it 
# since standard load_dotenv might not handle 'export' and the specific format correctly.
def load_envrc(filepath):
    if os.path.exists(filepath):
        with open(filepath, 'r') as f:
            for line in f:
                line = line.strip()
                if line.startswith('export '):
                    # Remove 'export ' and handle both 'KEY="VALUE"' and 'KEY: "VALUE"' formats
                    part = line[7:]
                    if '=' in part:
                        key, value = part.split('=', 1)
                    elif ': ' in part:
                        key, value = part.split(': ', 1)
                    else:
                        continue
                    
                    # Strip quotes from key and value
                    key = key.strip().strip('"').strip("'")
                    value = value.strip().strip('"').strip("'")
                    os.environ[key] = value

load_envrc('.envrc')
print("OPENAI_API_KEY loaded:", "OPENAI_API_KEY" in os.environ)


OPENAI_API_KEY loaded: True


In [3]:
df = pd.read_parquet('data/dataset_math.parquet')
print(df.shape)
df.head()

(200035, 2)


,question,answer
0,Jungkook is the 5th place. Find the number of ...,"If Jungkook is in 5th place, then 4 people cro..."
1,A number divided by 10 is 6. Yoongi got the re...,"Let's call the certain number ""x"". According t..."
2,Dongju selects a piece of paper with a number ...,To find the second smallest and third smallest...
3,"You wanted to subtract 46 from a number, but y...",If you accidentally subtracted 59 instead of 4...
4,The length of one span of Jinseo is about 12 c...,If one span of Jinseo is about 12 centimeters ...


In [5]:

def build_prompt(question: str, cot: bool = False) -> str:
    if cot:
        return f"Q: {question}\nSolve the problem, think step by step and give your answer."
    else:
        return f"Q: {question}\nA:"


In [ ]:
# OpenAI API call
from openai import OpenAI
import os

# Ensure you have set your OPENAI_API_KEY environment variable
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

def run_llm(prompt):
    response = client.chat.completions.create(
        model="gpt-5.1",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )
    return response.choices[0].message.content

In [ ]:
# # Example of Google Gemini API call
# import google.generativeai as genai
# import os

# # Ensure you have set your GOOGLE_API_KEY environment variable
# genai.configure(api_key=os.environ.get("GOOGLE_API_KEY"))

# def run_llm(prompt):
#     model = genai.GenerativeModel("gemini-1.5-flash")
#     response = model.generate_content(
#         prompt,
#         generation_config=genai.types.GenerationConfig(
#             temperature=0
#         )
#     )
#     return response.text

In [7]:
# Limit to top 20 for testing
eval_df = df.head(20).copy()

# Add LLM output to the dataframe
eval_df['LLM_output'] = eval_df['question'].apply(lambda q: run_llm(build_prompt(q, cot=True)))


In [8]:
eval_df.to_csv('data/dataset_math_with_llm_output.csv', index=False)

In [9]:
df_output = pd.read_csv('data/dataset_math_with_llm_output.csv')
df_output.head()

,question,answer,LLM_output
0,Jungkook is the 5th place. Find the number of ...,"If Jungkook is in 5th place, then 4 people cro...",To determine the number of people who crossed ...
1,A number divided by 10 is 6. Yoongi got the re...,"Let's call the certain number ""x"". According t...","To solve the problem, we need to find the numb..."
2,Dongju selects a piece of paper with a number ...,To find the second smallest and third smallest...,"To solve this problem, we need to determine al..."
3,"You wanted to subtract 46 from a number, but y...",If you accidentally subtracted 59 instead of 4...,"To solve this problem, let's break it down ste..."
4,The length of one span of Jinseo is about 12 c...,If one span of Jinseo is about 12 centimeters ...,"To solve the problem, we need to determine the..."
